## importing require libraries

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

from pyspark.sql import Window

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "payments", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")



In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show()


In [0]:
# ================================= QUALITY CHECK ======================

# 01 checking payment_method columm
df_bronze.select("payment_method").distinct().show()

# Checking null count in each column 
null_counts = df_bronze.select([F.count(F.when(F.isnull(c), c)).alias(c) for c in df_bronze.columns])
null_counts.show()

# 01 checking payment_status columm
df_bronze.select("payment_status").distinct().show()

## checking amount column
df_bronze.filter((F.col("amount") < 0) | (F.col("amount").isNull())).show()

## checking duplicate payment_id
df_bronze.groupBy("payment_id").count().filter(F.col("count") > 1).show()


In [0]:

# 1. Define the window: group by ID, order by the most recent time
window_spec = Window.partitionBy("payment_id").orderBy(F.col("payment_timestamp").desc())

# 2. Filter to keep only the latest record (Row 1)
df_silver = (
    df_bronze.withColumn("row_num", F.row_number().over(window_spec))
      .filter(F.col("row_num") == 1)
      .drop("row_num")
)
df_silver.show()


In [0]:
df_valid_amount = ((F.col("amount") > 0) &
    (F.col("amount").isNotNull()))

df_valid = df_silver.filter(df_valid_amount)
df_invalid = df_silver.filter(~df_valid_amount)
df_silver = df_valid


df_invalid.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.quarantine.bad_payment_amount"
    )

In [0]:

df_silver = df_silver.dropDuplicates(["payment_id"])
if not (spark.catalog.tableExists(f"{catalog}.{silver_schema}.{data_source}")):
    df_silver.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .mode("overwrite") \
    .saveAsTable(
        f"{catalog}.{silver_schema}.{data_source}"
    )
    print("sucessfully writed  data to delta location")
else:
    print("Doing upsert operation")
    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{silver_schema}.{data_source}"
    )

    delta_table.alias("target").merge(
        source=df_silver.alias("source"),
        condition="""
            target.payment_id = source.payment_id
        """
    ).whenMatchedUpdate(
        condition="""
        NOT (target.payment_method <=> source.payment_method)
        OR NOT (target.payment_status <=> source.payment_status)
        OR NOT (target.amount <=> source.amount)
        OR NOT (target.payment_timestamp <=> source.payment_timestamp)
    """,
        set={
             "payment_method": "coalesce(source.payment_method, target.payment_method)",
            "payment_status": "coalesce(source.payment_status, target.payment_status)",
            "amount": "coalesce(source.amount, target.amount)",
            "payment_timestamp": "coalesce(source.payment_timestamp, target.payment_timestamp)"
            
        }
    ).whenNotMatchedInsert(
        values={
            "payment_id": "source.payment_id",
            "payment_method": "source.payment_method",
            "payment_status": "source.payment_status",
            "amount": "source.amount",
            "payment_timestamp": "source.payment_timestamp"
        }
    ).execute()

